## SQL — Análise de livros, editoras, autores, avaliações e resenhas

## Objetivo do estudo

O objetivo deste projeto é analisar um banco de dados de um serviço de livros, utilizando consultas SQL, para entender melhor o comportamento do catálogo, das editoras, dos autores e dos usuários.

A análise será usada para gerar insights que possam ajudar na criação de uma proposta de valor para um novo produto voltado para pessoas que gostam de leitura.

In [2]:
import pandas as pd
from sqlalchemy import create_engine

db_config = {
    'user': 'YOUR_USERNAME',
    'pwd': 'YOUR_PASSWORD',
    'host': 'YOUR_HOST',
    'port': 5432,
    'db': 'YOUR_DATABASE'
}


connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(
    connection_string,
    connect_args={'sslmode': 'require'}
)

In [2]:
def run_query(query):
    return pd.read_sql(query, con=engine)

Para deixar o notebook mais organizado, criei uma função chamada `run_query`. Ela recebe uma consulta SQL e retorna o resultado em formato de tabela usando Pandas apenas para visualização.

## Estudo das Tabelas

### Tabela Books

In [3]:
query = '''
SELECT *
FROM books
LIMIT 5;
'''

run_query(query)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268




A tabela `books` reúne as informações principais dos livros, incluindo título, autor, editora, número de páginas e data de publicação.

Ela será a tabela central da análise, pois se relaciona com autores, editoras, avaliações e resenhas.

### Tabela Authors

In [4]:
query = '''
SELECT *
FROM authors
LIMIT 5;
'''

run_query(query)

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd




A tabela `authors` armazena as informações dos autores e será utilizada para relacionar cada livro ao seu respectivo autor durante as análises.

### Tabela Publishers


In [5]:
query = '''
SELECT *
FROM publishers
LIMIT 5;
'''

run_query(query)

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company



A tabela `publishers` contém os identificadores e os nomes das editoras responsáveis pelas publicações dos livros.

### Tabela Ratings

In [6]:
query = '''
SELECT *
FROM ratings
LIMIT 5;
'''

run_query(query)

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2




A tabela `ratings` registra as avaliações dos usuários, permitindo analisar a percepção dos leitores sobre cada livro.

### Tabela Reviews

In [7]:
query = '''
SELECT *
FROM reviews
LIMIT 5;
'''

run_query(query)

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...




A tabela `reviews` contém as resenhas escritas pelos usuários, permitindo analisar comentários e opiniões sobre os livros do catálogo.

### Contagem de Registros

In [8]:
query = '''
SELECT
    (SELECT COUNT(*) FROM books) AS total_books,
    (SELECT COUNT(*) FROM authors) AS total_authors,
    (SELECT COUNT(*) FROM publishers) AS total_publishers,
    (SELECT COUNT(*) FROM ratings) AS total_ratings,
    (SELECT COUNT(*) FROM reviews) AS total_reviews;
'''

run_query(query)

,total_books,total_authors,total_publishers,total_ratings,total_reviews
0,1000,636,340,6456,2793


### Visão geral dos dados

O banco de dados contém 1.000 livros, escritos por 636 autores e publicados por 340 editoras.

Além disso, há 6.456 avaliações e 2.793 resenhas registradas pelos usuários, indicando um volume significativo de interações que permitirá analisar a popularidade e a percepção dos leitores em relação aos livros disponíveis no catálogo.

## 1. Número de livros publicados após 1º de janeiro de 2000

Nesta etapa, será calculada a quantidade de livros publicados após 1º de janeiro de 2000, permitindo avaliar a representatividade de obras mais recentes no catálogo.

In [9]:
query = '''
SELECT
    COUNT(*) AS total_books_after_2000
FROM books
WHERE publication_date > '2000-01-01';
'''

run_query(query)

,total_books_after_2000
0,819




Conluimos que dos 1.000 livros presentes no banco de dados, 819 foram publicados após 1º de janeiro de 2000.

Isso indica que aproximadamente 82% do catálogo é composto por obras publicadas no século XXI, demonstrando uma predominância de títulos mais recentes na plataforma.

## 2. Número de avaliações e classificação média para cada livro

Nesta etapa, será calculado o número de avaliações e a classificação média de cada livro para identificar quais obras receberam maior participação dos usuários e como elas foram avaliadas.

In [10]:
query = '''
SELECT
    b.book_id,
    b.title,
    COUNT(DISTINCT rt.rating_id) AS rating_count,
    ROUND(AVG(rt.rating), 2) AS avg_rating
FROM books AS b
LEFT JOIN ratings AS rt
    ON b.book_id = rt.book_id
GROUP BY
    b.book_id,
    b.title
ORDER BY
    rating_count DESC;
'''

run_query(query)

,book_id,title,rating_count,avg_rating
0,948,Twilight (Twilight #1),160,3.66
1,750,The Hobbit or There and Back Again,88,4.13
2,673,The Catcher in the Rye,86,3.83
3,75,Angels & Demons (Robert Langdon #1),84,3.68
4,302,Harry Potter and the Prisoner of Azkaban (Harr...,82,4.41
...,...,...,...,...
995,175,Dance Dance Dance,2,4.50
996,403,Lord John and the Private Matter (Lord John Gr...,2,3.50
997,400,London Bridges (Alex Cross #10),2,4.00
998,401,Long Day's Journey into Night,2,4.00




Os resultados mostram que alguns livros concentram um volume significativamente maior de avaliações do que outros.

Entre os títulos com maior participação dos usuários, destacam-se *Twilight*, *The Hobbit*, *The Catcher in the Rye*, *Angels & Demons* e *Harry Potter and the Prisoner of Azkaban*.

Observa-se também que livros com muitas avaliações nem sempre apresentam as maiores classificações médias, indicando que popularidade e satisfação dos leitores nem sempre caminham juntas.

In [11]:
query = '''
SELECT
    b.book_id,
    b.title,
    COUNT(rt.rating_id) AS rating_count,
    ROUND(AVG(rt.rating), 2) AS avg_rating
FROM books AS b
LEFT JOIN ratings AS rt
    ON b.book_id = rt.book_id
GROUP BY
    b.book_id,
    b.title
ORDER BY
    avg_rating DESC,
    rating_count DESC
LIMIT 10;
'''

run_query(query)

,book_id,title,rating_count,avg_rating
0,17,A Dirty Job (Grim Reaper #1),4,5.0
1,553,School's Out—Forever (Maximum Ride #2),4,5.0
2,444,Moneyball: The Art of Winning an Unfair Game,3,5.0
3,347,In the Hand of the Goddess (Song of the Liones...,3,5.0
4,418,March,2,5.0
5,390,Light in August,2,5.0
6,224,Evening Class,2,5.0
7,182,Dead Souls,2,5.0
8,972,Wherever You Go There You Are: Mindfulness Me...,2,5.0
9,57,Act of Treason (Mitch Rapp #9),2,5.0




Observação complementar ao analisar os livros com maiores médias de avaliação, observa-se que muitos deles possuem poucas classificações. Isso sugere que uma nota média elevada nem sempre representa consenso entre os leitores, sendo importante considerar também a quantidade de avaliações recebidas.

## 3. Editora que publicou mais livros com mais de 50 páginas

Nesta etapa, vou identificar a editora com maior quantidade de livros com mais de 50 páginas. Esse filtro é importante para excluir publicações muito curtas, como brochuras ou folhetos.

In [12]:
query = '''
SELECT
    p.publisher,
    COUNT(b.book_id) AS books_count
FROM books AS b
INNER JOIN publishers AS p ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY
    p.publisher
ORDER BY
    books_count DESC
LIMIT 1;
'''

run_query(query)

,publisher,books_count
0,Penguin Books,42



Concluimos que entre as editoras presentes no banco de dados, a Penguin Books se destacou com 42 livros contendo mais de 50 páginas.

Esse resultado demonstra uma participação expressiva da editora no catálogo e sugere que ela possui um portfólio diversificado de obras, podendo ser considerada uma das editoras mais relevantes da plataforma.

## 4. Autor com a maior média de classificação

Nesta etapa, será identificado o autor com a maior média de classificação, considerando apenas livros que receberam pelo menos 50 avaliações. Esse critério ajuda a tornar o resultado mais confiável, reduzindo o impacto de livros com poucas avaliações.

In [13]:
query = '''
WITH books_with_50_ratings AS (
    SELECT
        book_id,
        COUNT(rating_id) AS rating_count,
        AVG(rating) AS avg_rating
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating_id) >= 50
)

SELECT
    a.author,
    ROUND(AVG(bwr.avg_rating), 2) AS avg_author_rating
FROM books_with_50_ratings AS bwr
INNER JOIN books AS b ON bwr.book_id = b.book_id
INNER JOIN authors AS a ON b.author_id = a.author_id
GROUP BY
    a.author
ORDER BY
    avg_author_rating DESC
LIMIT 1;
'''

run_query(query)

,author,avg_author_rating
0,J.K. Rowling/Mary GrandPré,4.28




Considerando apenas livros com pelo menos 50 avaliações, o autor J.K. Rowling/Mary GrandPré apresentou a maior média de classificação, com nota média de 4,28.

Esse resultado demonstra que as obras associadas à autora mantêm um alto nível de aprovação entre os leitores, mesmo quando analisadas apenas entre livros com um volume significativo de avaliações. Isso sugere uma combinação de popularidade e satisfação dos usuários, tornando suas obras referências importantes dentro do catálogo.

## 5. Número médio de resenhas entre usuários que avaliaram mais de 50 livros

Nesta etapa, será calculada a quantidade média de resenhas escritas por usuários que avaliaram mais de 50 livros. O objetivo é entender o nível de engajamento dos leitores mais ativos da plataforma.

In [14]:
query = '''
WITH active_users AS (
    SELECT
        username
    FROM ratings
    GROUP BY username
    HAVING COUNT(rating_id) > 50
)

SELECT
    ROUND(AVG(review_count), 2) AS avg_reviews
FROM (
    SELECT
        r.username,
        COUNT(rv.review_id) AS review_count
    FROM active_users r
    LEFT JOIN reviews rv
        ON r.username = rv.username
    GROUP BY r.username
) t;
'''

run_query(query)

,avg_reviews
0,24.33




Os usuários que avaliaram mais de 50 livros escreveram, em média, 24,33 resenhas.

Esse resultado mostra que os usuários mais ativos não apenas atribuem notas aos livros, mas também contribuem com comentários escritos. Esse grupo pode ser considerado importante para a plataforma, pois ajuda a aumentar o engajamento e a qualidade das informações disponíveis para outros leitores.

## Conclusão Final

## Conclusão Final

Ao longo desta análise, foi possível explorar diferentes aspectos do banco de dados e compreender melhor a relação entre livros, autores, editoras e usuários da plataforma.

Os resultados mostraram que o catálogo é composto majoritariamente por obras publicadas nas últimas décadas, além de apresentar um volume significativo de avaliações e resenhas realizadas pelos leitores. Isso demonstra que os usuários não apenas consomem conteúdo, mas também participam ativamente da plataforma por meio de suas opiniões e avaliações.

A análise também permitiu identificar autores e editoras que se destacam dentro do catálogo, além de mostrar que a popularidade de um livro nem sempre está diretamente relacionada à sua avaliação média. Esse tipo de informação pode contribuir para uma compreensão mais ampla das preferências dos leitores e auxiliar na construção de recomendações mais relevantes.

De forma geral, o banco de dados apresenta informações valiosas para o desenvolvimento de um aplicativo voltado para amantes da leitura. Os insights obtidos podem apoiar funcionalidades como recomendações personalizadas, destaque para livros e autores bem avaliados e estratégias para incentivar a participação dos usuários por meio de avaliações e resenhas.

Além disso, os resultados reforçam a importância de considerar diferentes indicadores em conjunto, já que a combinação entre características do catálogo e interação dos usuários oferece uma visão mais completa sobre o comportamento dos leitores e sobre as oportunidades de melhoria da experiência na plataforma.